# 06.1 — Multi-task losses overview

The Optimal model doesn't minimize *one* loss — it minimizes a weighted sum of **four**: reconstruction (does the decoder rebuild the input?), KL (is the latent well-behaved?), classification (are the predictions right?), and confidence (does the model know when it's unsure?). Orchestrating those four — computing each, normalizing them to a common scale, weighting, and summing into one number to `backward()` — is Module 06's subject. This notebook is the map; the following twelve are the territory.

This notebook covers:

* The four loss components and what each measures.
* Why they need *normalization* before summing (scales differ by orders of magnitude).
* The single-total-loss principle (one `backward()`, Critical Note #28).
* A tour of the Module 06 notebooks and how they fit together.

**Prerequisites:** [02.5 autograd](../02_numpy_and_pytorch_basics/02.5_autograd_basics.ipynb), [04.7 weighted loss](../04_architecture/04.7_weighted_classification_loss.ipynb), [05.1 the loop](../05_training_loop/05.1_the_custom_training_loop.ipynb).

## Section 1 — What MATLAB does

`cgg_lossELBO_v2` and its callers assemble the total loss from components, each with a weight, normalized against a running reference:

```matlab
Loss_Reconstruction = 0.5 * l2loss(Yrec, T, 'Mask', ~isnan(T));
Loss_KL             = klLoss(mu, logSigmaSq);
Loss_Classification = crossentropy(Yclass, Tclass, 'Weights', classWeights);
Loss_Confidence     = confidenceLoss(...);

% Normalize each to Classification's scale (EMA priors), then weight and sum:
Loss_Total = WeightRec * norm(Loss_Reconstruction) ...
           + WeightKL  * norm(Loss_KL) ...
           + WeightClass * Loss_Classification ...
           + WeightConf * norm(Loss_Confidence);
```

The `norm(...)` step — EMA prior normalization — is the non-obvious part: the four losses live on wildly different scales, and summing them raw would let one dominate purely by magnitude. Module 06 unpacks every piece of this.

## Section 2 — The Python concepts you need

### 2.1 — The four components

| Component | Measures | Notebook |
|---|---|---|
| **Reconstruction** | decoder rebuilds the (NaN-masked) input | [06.2](06.2_vae_and_the_elbo.ipynb), [06.10](06.10_nan_masked_reconstruction.ipynb) |
| **KL** | latent posterior ≈ unit Gaussian | [06.2](06.2_vae_and_the_elbo.ipynb) |
| **Classification** | per-dimension predictions correct | [04.7](../04_architecture/04.7_weighted_classification_loss.ipynb), [06.5](06.5_mil_softmax_pooling.ipynb) |
| **Confidence** | model's self-assessed certainty is calibrated | [06.6](06.6_confidence_routing.ipynb), [06.7](06.7_the_confidence_pd_controller.ipynb) |

Reconstruction + KL together form the **ELBO** (the VAE objective); classification + confidence are the supervised signal. All four are summed into one `total_loss`.

### 2.2 — Why raw sums fail: the scale problem

Compute a toy version of each and look at their magnitudes — they differ by orders of magnitude, so a naive sum is dominated by whichever happens to be largest:

In [ ]:
import torch

# Stand-in values on realistic scales:
loss_recon = torch.tensor(1200.0)     # sum over many channels × timesteps → large
loss_kl    = torch.tensor(3.5)        # per-latent, small
loss_class = torch.tensor(1.1)        # cross-entropy, order ~1
loss_conf  = torch.tensor(0.02)       # tiny

naive_total = loss_recon + loss_kl + loss_class + loss_conf
print(f"raw components: recon={loss_recon}, kl={loss_kl}, class={loss_class}, conf={loss_conf}")
print(f"naive sum: {naive_total}")
print(f"reconstruction is {100*loss_recon/naive_total:.1f}% of it — classification barely matters.")

Reconstruction is ~99% of the naive sum, so the gradient would optimize reconstruction and ignore classification — even though classification is the *point*. The fix is **EMA prior normalization** ([06.4](06.4_the_ema_prior_normalization_deep_dive.ipynb), [06.12](06.12_ema_prior_normalization_deep_dive.ipynb)): rescale each component to classification's magnitude using a running average, so the *weights* (`WeightRec`, `WeightKL`, …) control the balance rather than the accidental scales. Classification is the reference because it's the task that matters.

### 2.3 — One total loss, one backward

In [ ]:
# Weighted sum AFTER normalization is the single scalar we backprop:
w_recon, w_kl, w_class, w_conf = 100.0, 1.0, 10.0, 1.0
# (imagine each component already normalized to ~1.0 scale)
norm_recon, norm_kl, norm_class, norm_conf = 1.0, 1.0, 1.0, 1.0

total_loss = (w_recon * norm_recon + w_kl * norm_kl +
              w_class * norm_class + w_conf * norm_conf)
print(f"total_loss (one scalar): {total_loss}")
print("This ONE number gets total_loss.backward() — autograd distributes its")
print("gradient to the encoder, decoder, AND classifier through the graph (02.5 §3).")

This is Critical Note #28, taught fully in [06.11](06.11_single_total_loss_three_subnetworks.ipynb): even though the loss sums four components touching three subnetworks, there is exactly **one** `backward()` on one scalar. The graph does the routing — you never compute per-subnetwork gradients by hand (which MATLAB's `dlgradient` plumbing had to). The intermediate `Loss_Decoder`, `Loss_Classifier` sums are just numbers on the way to `total_loss`; they are NOT separate gradient roots.

### 2.4 — The Module 06 map

| Notebook | Topic |
|---|---|
| 06.1 (here) | the overview |
| 06.2 | VAE + ELBO — reconstruction + KL, the sampling layer |
| 06.3 | stochastic vs deterministic sampling placement |
| 06.4 / 06.12 | EMA prior normalization (the scale fix from §2.2) |
| 06.5 | MIL softmax pooling |
| 06.6 | confidence routing (Trial vs Task) |
| 06.7 | the confidence PD-controller — **the highest-risk port** |
| 06.8 | L2 inside the loss kernel (why ≠ `weight_decay`) |
| 06.9 | per-batch prior correction |
| 06.10 | NaN-masked reconstruction |
| 06.11 | single total loss → three subnetworks |
| 06.13 | the sampling layer's deterministic-at-inference behavior |

Read them roughly in order; 06.2 and 06.4 are the load-bearing foundations.

## Section 3 — The `neural_data_decoding` implementation

`aggregate_total_loss` — where the four components become one scalar:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/training/losses/multi_objective.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if line.startswith("def aggregate_total_loss"))
# print the signature + docstring opening
for k in range(i, min(i + 12, len(src))):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* The function takes each component's loss and a weight dict, returning a `LossBreakdown` — the aggregation is one place, not scattered.
* `aggregate_normalized_losses` (further down the file) adds the EMA prior normalization layer from §2.2 — normalization and weighting are separate, composable steps.
* The extra keys in the weight dict are forwarded harmlessly ([05.1 §3](../05_training_loop/05.1_the_custom_training_loop.ipynb)) — a Milestone-A run with only `{"classification": w}` uses the same aggregator as the full Optimal run.
* The breakdown carries per-component values for *monitoring* — you can watch reconstruction vs classification separately even though only their sum is backpropped.

## Section 4 — Hands-on exercises

### Exercise 4.1 — the domination effect

Take the §2.2 raw components and show that halving the (already-tiny) classification loss barely moves the naive total, while halving reconstruction moves it a lot — the asymmetry normalization fixes.

In [ ]:
# Reveal:
base = loss_recon + loss_kl + loss_class + loss_conf
half_class = loss_recon + loss_kl + loss_class/2 + loss_conf
half_recon = loss_recon/2 + loss_kl + loss_class + loss_conf
print(f"halving classification changes total by {abs(base-half_class):.2f}")
print(f"halving reconstruction changes total by {abs(base-half_recon):.2f}")
print("→ raw sum is ~all reconstruction; the task loss is invisible to the gradient.")

### Exercise 4.2 — one backward, three subnetworks

Build a toy composite (encoder → classifier), a total loss summing two components, and confirm ONE `backward()` populates gradients in BOTH subnetworks.

In [ ]:
# Reveal:
import torch.nn as nn
enc = nn.Linear(8, 4); clf = nn.Linear(4, 3)
x = torch.randn(5, 8)
latent = enc(x)
recon_loss = (latent ** 2).mean()                      # a fake "reconstruction"
class_loss = nn.functional.cross_entropy(clf(latent), torch.tensor([0,1,2,0,1]))
(100 * recon_loss + 10 * class_loss).backward()        # ONE backward on the sum
print("encoder got gradients?   ", enc.weight.grad is not None)
print("classifier got gradients?", clf.weight.grad is not None)

## Section 5 — Common errors

### One component dominates; others don't improve

The scale problem (§2.2) — raw-summed losses on different magnitudes. Normalize before summing ([06.4](06.4_the_ema_prior_normalization_deep_dive.ipynb)); the weights should control balance, not accidental scale.

### Calling backward() per component

Wasteful and wrong for shared parameters — the graph would be traversed multiple times (and freed after the first). Sum into one scalar, one `backward()` (§2.3, [06.11](06.11_single_total_loss_three_subnetworks.ipynb)).

### A weight of 0 doesn't disable a component cleanly

A zero weight zeroes the contribution but the component is still *computed* (wasted work) and can still produce NaN that poisons the sum ([02.8 §2.2](../02_numpy_and_pytorch_basics/02.8_nan_handling.ipynb)). Genuinely disabling a component means skipping its computation, which the composite does for absent confidence heads.

### Monitoring the wrong number

The `total_loss` is one number; to see *which* component is improving, log the breakdown's per-component values. A flat total can hide reconstruction improving while classification stalls.

### Weights that worked in MATLAB give different balance in Python

The normalization reference or the component scales differ. The weights act on the *post-normalization* scale ([06.4](06.4_the_ema_prior_normalization_deep_dive.ipynb)) — check that normalization is active and using the same reference (classification).

## Section 6 — Further reading

- [Multi-task learning survey](https://arxiv.org/abs/1706.05098) — loss weighting and balancing strategies.
- [`src/neural_data_decoding/training/losses/multi_objective.py`](../../src/neural_data_decoding/training/losses/multi_objective.py) — the aggregator.
- The Module 06 notebooks (§2.4) — each component in depth.

Next up: **[06.2 VAE and the ELBO](06.2_vae_and_the_elbo.ipynb)** — reconstruction + KL, the reparameterization trick, and the sampling layer.